# RFSoC-PYNQ Multi-Tile Synchronisation with Digital Feature Sync.
---

<div class="alert alert-box alert-info">
Please use Jupyter Labs http://&lt;board_ip_address&gt;/lab for this notebook.
</div>


This notebook is a stripped back version of the demonstrator notebook and shows how the required RF-ADCs and RF-DACs can be synchronised during initialisation of ```Overlay```. For further details on the MTS system and the overall design, please refer to the [demonstrator notebook](02_rfsoc_mts_demonstrator.ipynb).

----

### Overlay Setup <a class="anchor" id="waveform-gen"></a>

Synchronisation of the RFDCs at boot time is enabled by initialising the ```Overlay``` class with the keyword argument ```sync_on_init=True```. This argument is then passed to the ```Overlay```’s ```MultiTileSynchronisation``` object. When set to ```True```, the sequence of steps described in the demonstrator notebook is executed within the ```__init__()``` method, resulting in the RFDCs being synchronised during initialisation.

In [1]:
from rfsoc_mts.overlay import Overlay
import numpy as np

ol = Overlay(bitfile_name=None, # Default bitstream
             lmx_freq=500,
             lmk_freq=500,
             dac_mts_reftile=2,
             adc_mts_reftile=2,
             dac_mts_tiles=0b0101,
             adc_mts_tiles=0b0101,
             sync_on_init=True)

### Waveform Generation <a class="anchor" id="waveform-gen"></a>
As before, the waveform transmitted by the RF-DAC is generated in the PS. The following cell selects and characterises a waveform from ```{'Sinusoid', 'Chirp'}```. The initial NCO frequencies of the RFDCs are also configured at this stage.

In this case, however, the ```ol.configure_rfdcs()``` method is not used. Instead, the ```nco_dac``` and ```nco_adc``` properties of the ```MultiTileSynchronisation``` object are used to set the initial NCO frequencies.

In [2]:
# -- Characterise waveform --
Fs = 500.0e6                         # Baseband sampling rate produced by signal generator - determined by the hardware design.
N = int(ol.signal_player.length/2)   # Signal Player buffer length

DAC_Amplitude = 2**14-1              # Scaling by the fractional bits
Fc = 1000                            # NCO frequency

offset = 12.5                        # NCO offset for sinusoid case - frequency of downconverted signal observed
chirp_range = [-80e6, 80e6]          # Chirp frequency range

sig_type = 'Sinusoid'

ol.mts.nco_dac = Fc
ol.mts.nco_adc = Fc


# -- Generate waveform --
if sig_type == 'Sinusoid':
    signal = DAC_Amplitude * (1+1j*0)*np.ones(N)
    ol.mts.nco_adc = Fc-offset       # Apply ADC frequency offset so sinusoid can be observed after DDC
    
elif sig_type == 'Chirp':
    t = np.arange(ol.signal_player.length/2, dtype=np.single) / Fs
    window = np.hanning(ol.signal_player.length/2)
    
    k = (chirp_range[1] - chirp_range[0]) / t[-1]
    phase = 2*np.pi*(chirp_range[0]*t + 0.5*k*(t**2))
    signal = DAC_Amplitude * (window * np.exp(1j*phase)).astype(np.complex64)
    
else:
    raise ValueError(f"Unsupported signal type: {sig_type}")

With the signal generated, we can load it into the Signal Player's internal memory, and enable iterative BRAM reads to feed the RF-DAC with the waveform.

In [3]:
# -- Disable signal output --
ol.signal_player.disable()

# -- Load signal and re-enable output --
ol.signal_player.load_signal(signal)
ol.signal_player.enable()

The final step is to synchronously capture the samples of each RF-ADC by triggering the GPIO in Signal Capture, reading the BRAMs, and plotting the resulting signal using ```ol.plot_adc()```.

In [4]:
ol.plot_adc(
    sample_matrix=ol.signal_capture.capture_adc(),
    window=128,
    title="Multi-Tile Synchronisation"
)

NameError: name 'plot_adc' is not defined

The plot above should demonstrate that, for the loopback connected data converters, the MTS engine has successfully synchronised both RF-DAC and RF-ADCs.